# Clase 7 - Unidad 3
Aplicaciones Computacionales en Ing.

* Franco Mansilla Ibáñez (www.francomansilla.com)

In [1]:
import pandas as pd
import numpy as np

In [2]:
dir = "/Users/francomansilla/Library/CloudStorage/GoogleDrive/Mi unidad/universidad mayor/Aplicaciones Computacionales/Clase 6"

In [3]:
file_path = dir + "/base de datos autos.xls"

# Leer el archivo Excel
df = pd.read_excel(file_path)

In [4]:
df.head()

,make,price,mpg,rep78,weight,length,foreign
0,AMC Concord,4099,22,3.0,2930,186,Domestic
1,AMC Pacer,4749,17,3.0,3350,173,Domestic
2,AMC Spirit,3799,22,NaN,2640,168,Domestic
3,Buick Century,4816,20,3.0,3250,196,Domestic
4,Buick Electra,7827,15,4.0,4080,222,Domestic


# 1. Exploración de Datos

* Valores unicos

In [5]:
df['rep78'].unique()

array([ 3., nan,  4.,  2.,  5.,  1.])

In [6]:
df['foreign'].unique()

array(['Domestic', 'Foreign'], dtype=object)

In [7]:
df['foreign'].nunique()

2

* Conteo por grupos

In [8]:
df['foreign'].value_counts()

foreign
Domestic    52
Foreign     22
Name: count, dtype: int64

In [9]:
count_foreign = df['rep78'].value_counts()
count_foreign

rep78
3.0    30
4.0    18
5.0    11
2.0     8
1.0     2
Name: count, dtype: int64

* Tablas cruzadas

In [10]:
pd.crosstab(df['rep78'], df['foreign'])

foreign,Domestic,Foreign
rep78,,
1.0,2,0
2.0,8,0
3.0,27,3
4.0,9,9
5.0,2,9


In [11]:
pd.crosstab(df['rep78'], df['foreign'], margins=True)

foreign,Domestic,Foreign,All
rep78,,,
1.0,2,0,2
2.0,8,0,8
3.0,27,3,30
4.0,9,9,18
5.0,2,9,11
All,48,21,69


In [12]:
pd.crosstab(df['rep78'], df['foreign'], values=df['price'], aggfunc='mean')

foreign,Domestic,Foreign
rep78,,
1.0,4564.500000,NaN
2.0,5967.625000,NaN
3.0,6607.074074,4828.666667
4.0,5881.555556,6261.444444
5.0,4204.500000,6292.666667


In [13]:
tabla_cruzada = pd.crosstab(df['rep78'], df['foreign'], values=df['price'], aggfunc='max')
tabla_cruzada

foreign,Domestic,Foreign
rep78,,
1.0,4934.0,NaN
2.0,14500.0,NaN
3.0,15906.0,6295.0
4.0,8814.0,9735.0
5.0,4425.0,11995.0


* Crear variables por tramos

In [14]:
bins = [df['mpg'].min(),df['mpg'].mean(), df['mpg'].max()]  
labels = ['Bajo', 'Alto']
df['mpg_tramos'] = pd.cut(df['mpg'], bins=bins, labels=labels)
df.head()

,make,price,mpg,rep78,weight,length,foreign,mpg_tramos
0,AMC Concord,4099,22,3.0,2930,186,Domestic,Alto
1,AMC Pacer,4749,17,3.0,3350,173,Domestic,Bajo
2,AMC Spirit,3799,22,NaN,2640,168,Domestic,Alto
3,Buick Century,4816,20,3.0,3250,196,Domestic,Bajo
4,Buick Electra,7827,15,4.0,4080,222,Domestic,Bajo


In [15]:
tabla_cruzada = pd.crosstab(index=df['rep78'],
                            columns=[df['foreign'], df['mpg_tramos']],
                            values=df['price'],
                            aggfunc='mean')
tabla_cruzada

foreign    Domestic               Foreign        
mpg_tramos     Bajo         Alto     Bajo    Alto
rep78                                            
1.0         4934.00  4195.000000      NaN     NaN
2.0         6959.60  4314.333333      NaN     NaN
3.0         6613.45  4206.200000  4296.00  5095.0
4.0         6388.00  4109.000000  8129.00  6028.0
5.0             NaN  4204.500000  8325.75  4666.2

In [16]:
tabla_cruzada.style.format("{:.2f}")

# 2. Identificación y Tratamiento de Datos Atípicos

### Identificación
* Paso 1. Guardar en un diccionario media, mediana, mínimo, p1, p99, máximo, variación
* Paso 2. Si la variación porcentual es mayor al 15% se hace tratamiento. 

### Tratamiento.
* Paso 3. Si la variable tratada hay que winsorizar a percentil 1% y 99%
---

* **Identificación:** Paso 1

In [17]:
def calcular_estadisticas(columna):
    media = columna.mean()
    mediana = columna.median()
    minimo = columna.min()
    p1 = columna.quantile(0.01)
    p99 = columna.quantile(0.99)
    maximo = columna.max()
    variacion = (media - mediana) / media
    return {
        'media': media,
        'mediana': mediana,
        'minimo': minimo,
        'p1': p1,
        'p99': p99,
        'maximo': maximo,
        'variacion': variacion
    }

* **Identificación:** Paso 2

In [18]:
# Diccionario para guardar los resultados
resultados = {}

# Calcular estadísticas para cada columna numérica
for columna in df[['price', 'mpg', 'weight', 'length']]:
    resultados[columna] = calcular_estadisticas(df[columna])

# Mostrar el diccionario de resultados
resultados


{'price': {'media': 6165.256756756757,
  'mediana': 5006.5,
  'minimo': 3291,
  'p1': 3296.84,
  'p99': 14879.619999999995,
  'maximo': 15906,
  'variacion': 0.187949472742855},
 'mpg': {'media': 21.2972972972973,
  'mediana': 20.0,
  'minimo': 12,
  'p1': 12.0,
  'p99': 36.619999999999976,
  'maximo': 41,
  'variacion': 0.06091370558375639},
 'weight': {'media': 3019.4594594594596,
  'mediana': 3190.0,
  'minimo': 1760,
  'p1': 1789.2,
  'p99': 4752.4,
  'maximo': 4840,
  'variacion': -0.056480486931614705},
 'length': {'media': 187.93243243243242,
  'mediana': 192.5,
  'minimo': 142,
  'p1': 145.65,
  'p99': 230.81,
  'maximo': 233,
  'variacion': -0.024304307183432867}}

* **Tratamiento:** Paso 1 

In [19]:
def ajustar_valores(columna, columna_nombre):
    
    stats = resultados[columna_nombre]
    p1 = stats['p1']
    p99 = stats['p99']
    variacion = stats['variacion']
    outliers = []
    
    if variacion > np.abs(0.15):
        nuevos_valores = []
        for valor in columna:
            if valor > p99:
                nuevos_valores.append(p99)
                outliers.append(valor)
            elif valor < p1:
                nuevos_valores.append(p1)
                outliers.append(valor)
            else:
                nuevos_valores.append(valor)
                
        columna = pd.Series(nuevos_valores, index=columna.index)
    
    # Guardamos los outliers en el diccionario
    resultados[columna_nombre]['outliers'] = outliers
    return columna

In [20]:
for columna in df[['price', 'mpg', 'weight', 'length']]:
    df[columna] = ajustar_valores(df[columna], columna)

In [21]:
resultados

{'price': {'media': 6165.256756756757,
  'mediana': 5006.5,
  'minimo': 3291,
  'p1': 3296.84,
  'p99': 14879.619999999995,
  'maximo': 15906,
  'variacion': 0.187949472742855,
  'outliers': [15906, 3291]},
 'mpg': {'media': 21.2972972972973,
  'mediana': 20.0,
  'minimo': 12,
  'p1': 12.0,
  'p99': 36.619999999999976,
  'maximo': 41,
  'variacion': 0.06091370558375639,
  'outliers': []},
 'weight': {'media': 3019.4594594594596,
  'mediana': 3190.0,
  'minimo': 1760,
  'p1': 1789.2,
  'p99': 4752.4,
  'maximo': 4840,
  'variacion': -0.056480486931614705,
  'outliers': []},
 'length': {'media': 187.93243243243242,
  'mediana': 192.5,
  'minimo': 142,
  'p1': 145.65,
  'p99': 230.81,
  'maximo': 233,
  'variacion': -0.024304307183432867,
  'outliers': []}}

In [30]:
# Exportar Base de Datos tratada
df.to_excel("/Users/francomansilla/Library/CloudStorage/GoogleDrive-franco.andres.mansilla@gmail.com/Mi unidad/universidad mayor/Aplicaciones Computacionales/Clase 8/BD_tratada.xlsx", index = False)